# 00 Data Inspection

This notebook is used to inspect the raw December 2019 e-commerce event data before building the full ETL pipeline. A sample of 100,000 records is loaded from the dataset for initial inspection.

The goal of this notebook is to:
- load a small sample of the raw dataset
- verify the dataset structure
- inspect columns and data types
- confirm that the extraction step works correctly

This notebook is part of a larger workflow where:
- ETL scripts in `src/etl/` handle reusable data loading and preprocessing logic
- notebooks are used for inspection, validation, and analysis

## ETL Connection

The data loading logic is kept outside the notebook in the ETL layer to keep the workflow modular and reusable.

For this step, the notebook uses:
- `src/etl/extract.py` → contains the function used to load a sample from the compressed raw dataset

This keeps extraction logic separated from notebook-based inspection.

## Setup

In [1]:
import sys
from pathlib import Path

# Add the project root to the Python path
sys.path.append(str(Path().resolve().parent))

## Import Dependencies

In [2]:
from src.etl.extract import load_sample

## Load Sample Data

In [3]:
file_path = "../data/raw/2019-Dec.csv.gz"
df = load_sample(file_path)

## Preview Data

In [4]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,view,1005105,2232732093077520756,construction.tools.light,apple,1302.48,556695836,ca5eefc5-11f9-450c-91ed-380285a0bc80
1,2019-12-01 00:00:00 UTC,view,22700068,2232732091643068746,NaN,force,102.96,577702456,de33debe-c7bf-44e8-8a12-3bf8421f842a
2,2019-12-01 00:00:01 UTC,view,2402273,2232732100769874463,appliances.personal.massager,bosch,313.52,539453785,5ee185a7-0689-4a33-923d-ba0130929a76
3,2019-12-01 00:00:02 UTC,purchase,26400248,2053013553056579841,computers.peripherals.printer,NaN,132.31,535135317,61792a26-672f-4e61-9832-7b63bb1714db
4,2019-12-01 00:00:02 UTC,view,20100164,2232732110089618156,apparel.trousers,nika,101.68,517987650,906c6ca8-ff5c-419a-bde9-967ba8e2233e


## Dataset Structure

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     100000 non-null  object 
 1   event_type     100000 non-null  object 
 2   product_id     100000 non-null  int64  
 3   category_id    100000 non-null  int64  
 4   category_code  88110 non-null   object 
 5   brand          86971 non-null   object 
 6   price          100000 non-null  float64
 7   user_id        100000 non-null  int64  
 8   user_session   100000 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 6.9+ MB


## Null Check

In [6]:
df.isnull().sum()

event_time           0
event_type           0
product_id           0
category_id          0
category_code    11890
brand            13029
price                0
user_id              0
user_session         0
dtype: int64

## Event Type Distribution

In [7]:
df['event_type'].value_counts()

event_type
view        95437
cart         3487
purchase     1076
Name: count, dtype: int64

## Unique Sessions

In [8]:
df['user_session'].nunique()

21492

## Unique Users

In [9]:
df['user_id'].nunique()

17208

## ETL Design

The dataset is large (2.74 GB compressed), so it cannot be processed directly in memory in a single step.

The ETL pipeline will:
- load data in chunks
- apply necessary transformations
- save processed data in an efficient format (Parquet)

This approach ensures scalability, efficiency, and reusability.

## Transformation Plan

Based on initial inspection, the following transformations will be applied:

- Convert `event_time` to datetime format
- Handle missing values in `category_code` and `brand`
- Ensure correct data types for all columns
- Prepare data for session-level analysis

## Load Processed Dataset

In [11]:
import pandas as pd
df_parquet = pd.read_parquet("../data/processed/2019-Dec.parquet")

## Dataset Shape

In [12]:
df_parquet.shape

(67430759, 9)

## Dataset Overview

In [13]:
df_parquet.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00+00:00,view,1005105,2232732093077520756,construction.tools.light,apple,1302.48,556695836,ca5eefc5-11f9-450c-91ed-380285a0bc80
1,2019-12-01 00:00:00+00:00,view,22700068,2232732091643068746,None,force,102.96,577702456,de33debe-c7bf-44e8-8a12-3bf8421f842a
2,2019-12-01 00:00:01+00:00,view,2402273,2232732100769874463,appliances.personal.massager,bosch,313.52,539453785,5ee185a7-0689-4a33-923d-ba0130929a76
3,2019-12-01 00:00:02+00:00,purchase,26400248,2053013553056579841,computers.peripherals.printer,None,132.31,535135317,61792a26-672f-4e61-9832-7b63bb1714db
4,2019-12-01 00:00:02+00:00,view,20100164,2232732110089618156,apparel.trousers,nika,101.68,517987650,906c6ca8-ff5c-419a-bde9-967ba8e2233e


## Dataset Info

In [14]:
df_parquet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67430759 entries, 0 to 67430758
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[us, UTC]
 1   event_type     object             
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  object             
 5   brand          object             
 6   price          float64            
 7   user_id        int64              
 8   user_session   object             
dtypes: datetime64[us, UTC](1), float64(1), int64(3), object(4)
memory usage: 4.5+ GB


## Missing Values Check

In [15]:
missing_values = df_parquet.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

brand            8113492
category_code    7078121
user_session          21
dtype: int64

## Duplicate Check

In [16]:
df_parquet.duplicated().sum()

0

## Event Type Distribution

In [17]:
df_parquet['event_type'].value_counts()

event_type
view        62984129
cart         3284602
purchase     1162028
Name: count, dtype: int64

## Time Range of Data

In [18]:
df_parquet['event_time'].min(), df_parquet['event_time'].max()

(Timestamp('2019-12-01 00:00:00+0000', tz='UTC'),
 Timestamp('2019-12-31 23:59:59+0000', tz='UTC'))

## Cardinality Check

In [19]:
{
    "event_type": df_parquet['event_type'].nunique(),
    "brand": df_parquet['brand'].nunique(),
    "category_code": df_parquet['category_code'].nunique(),
    "user_id": df_parquet['user_id'].nunique(),
    "user_session": df_parquet['user_session'].nunique()
}

{'event_type': 3,
 'brand': 4637,
 'category_code': 135,
 'user_id': 4577232,
 'user_session': 15581360}